<a href="https://colab.research.google.com/github/riyanmandanna01/RAG-project/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SIMPLE RAG PROJECT - GOOGLE COLAB
# ============================================================

# 1. Install required libraries
!pip -q install sentence-transformers faiss-cpu transformers accelerate

# ============================================================
# 2. Import libraries
# ============================================================

import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ============================================================
# 3. Create a sample knowledge base
#    You can replace this with your own document later.
# ============================================================

documents = [
    """
    Artificial Intelligence (AI) is a field of computer science
    that focuses on creating systems capable of performing tasks
    that normally require human intelligence. These tasks include
    learning, reasoning, problem solving, understanding language,
    and recognizing images.
    """,

    """
    Machine Learning (ML) is a subset of Artificial Intelligence.
    Machine learning algorithms learn patterns from data and use
    those patterns to make predictions or decisions without being
    explicitly programmed for every task.
    """,

    """
    Deep Learning is a subset of Machine Learning that uses
    artificial neural networks with multiple layers. Deep learning
    is widely used for image recognition, speech recognition,
    natural language processing, and autonomous systems.
    """,

    """
    Natural Language Processing (NLP) is a branch of Artificial
    Intelligence that enables computers to understand, process,
    generate, and interact with human language.
    """,

    """
    Retrieval-Augmented Generation (RAG) combines information
    retrieval with a language model. First, relevant information
    is retrieved from a knowledge base. The retrieved information
    is then provided to a language model so that it can generate
    an answer based on the available context.
    """,

    """
    Large Language Models (LLMs) are neural network models trained
    on large amounts of text. They can perform tasks such as
    answering questions, summarization, translation, text
    generation, and code generation.
    """
]

print("Number of documents:", len(documents))

# ============================================================
# 4. Load embedding model
# ============================================================

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# ============================================================
# 5. Convert documents into embeddings
# ============================================================

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

print("Embedding shape:", document_embeddings.shape)

# ============================================================
# 6. Create FAISS vector database
# ============================================================

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(document_embeddings).astype("float32")
)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)

# ============================================================
# 7. Load the LLM
# ============================================================

print("\nLoading language model...")

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM loaded successfully.")

# ============================================================
# 8. Retrieval function
# ============================================================

def retrieve_documents(query, top_k=2):

    # Convert user query into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    # Search FAISS
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_docs = []

    for i in indices[0]:
        if i < len(documents):
            retrieved_docs.append(documents[i])

    return retrieved_docs


# ============================================================
# 9. RAG answer generation function
# ============================================================

def rag_answer(question, top_k=2):

    # Retrieve relevant documents
    retrieved_docs = retrieve_documents(
        question,
        top_k
    )

    # Combine retrieved information
    context = "\n\n".join(retrieved_docs)

    # Create prompt
    prompt = f"""
Answer the question using only the information provided
in the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    # Tokenize prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate answer
    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_docs


# ============================================================
# 10. Test the RAG system
# ============================================================

question = "What is Retrieval-Augmented Generation?"

answer, retrieved_docs = rag_answer(question)

print("\n==============================")
print("QUESTION")
print("==============================")
print(question)

print("\n==============================")
print("RETRIEVED DOCUMENTS")
print("==============================")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nDocument {i}:")
    print(doc.strip())

print("\n==============================")
print("GENERATED ANSWER")
print("==============================")
print(answer)


# ============================================================
# 11. Interactive Question Answering
# ============================================================

print("\n\n==========================================")
print("RAG CHAT")
print("Type 'exit' to stop.")
print("==========================================")

while True:

    question = input("\nAsk a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer, retrieved_docs = rag_answer(question)

    print("\nAnswer:")
    print(answer)

Number of documents: 6

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully.


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QUESTION
What is Retrieval-Augmented Generation?

RETRIEVED DOCUMENTS

Document 1:
Retrieval-Augmented Generation (RAG) combines information
    retrieval with a language model. First, relevant information
    is retrieved from a knowledge base. The retrieved information
    is then provided to a language model so that it can generate
    an answer based on the available context.

Document 2:
Large Language Models (LLMs) are neural network models trained
    on large amounts of text. They can perform tasks such as
    answering questions, summarization, translation, text
    generation, and code generation.

GENERATED ANSWER
combines information retrieval with a language model


RAG CHAT
Type 'exit' to stop.

Ask a question: define LLM

Answer:
Large Language Models

Ask a question: Define RAG

Answer:
Retrieval-Augmented Generation

Ask a question: define AIML

Answer:
a language model

Ask a question: What is GAN

Answer:
Machine Learning

Ask a question: define RNN

Answer:
Deep Le